[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_04_exercise.ipynb)

# Module 6 Exercise: Which Vision System Should We Trust?

**Notebook:** `06_04_exercise`

## The situation

A satellite-imagery company wants to automate land-cover classification.

Thousands of image tiles arrive every day. A model will assign one of 10 EuroSAT land-cover categories.

The company does **not** need you to invent another architecture.

It needs a recommendation:

> **Which vision system should we use, when should we trust it, and when should a human review the result?**

You will compare two systems built from ideas in this module:

### System A — Human-designed representation

\[
\text{image}
\rightarrow
\text{34 named features}
\rightarrow
\text{random forest}
\]

The features summarize:

- color;
- color distributions;
- coarse edges.

### System B — Pretrained representation

\[
\text{image}
\rightarrow
\text{pretrained MobileNetV2 features}
\rightarrow
\text{logistic regression}
\]

MobileNetV2 is frozen. We are **not** training a CNN from scratch.

Instead, we reuse a visual representation learned from ImageNet and train only a lightweight classifier on top.

## Your job

You will evaluate both systems on:

1. overall accuracy;
2. class-specific errors;
3. disagreements between systems;
4. robustness to altered images;
5. confidence and human-review thresholds.

Then you will recommend a deployment policy.

---

## Suggested timing

| Part | Approx. time |
| --- | ---: |
| Build and benchmark the two systems | 10–15 min |
| Error analysis | 10 min |
| Robustness stress test | 10–15 min |
| Human-review threshold | 10 min |
| Recommendation | 10 min |

Most of the code is provided.

Your work is in the **decisions and interpretation**.

## Before you run anything

Based on what you learned in the walkthroughs, make a prediction.

### Question 1

Which system do you expect to have higher clean-test accuracy, and why?

Which system do you expect to be easier to explain?

**Your answer:**

> Replace this text with your prediction before running the models.

## 0) Setup

The first run downloads:

- EuroSAT from Zenodo;
- pretrained MobileNetV2 ImageNet weights.

A GPU is helpful for extracting pretrained features, but there is no neural-network training loop in this exercise.

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import convolve, gaussian_filter

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from IPython.display import display

SEED = 1955
N_SAMPLES = 2000
BATCH_SIZE = 64

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 1) Load a smaller EuroSAT exercise sample

The walkthrough notebooks used 5,000 images.

For this exercise, we use 2,000 so the analysis runs quickly enough for class.

The sampling is still:

- deterministic;
- stratified;
- 70% train;
- 10% validation;
- 20% test.

The validation split is available if you want it, but the main exercise focuses on the test set.

In [ ]:
DATA_DIR = "assets/data"
ZIP_PATH = os.path.join(DATA_DIR, "EuroSAT_RGB.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "EuroSAT_RGB")
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading EuroSAT (~90 MB, one-time)...")
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

if not os.path.exists(EXTRACT_DIR):
    print("Extracting EuroSAT...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

EXPECTED_CLASSES = {
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
}

def find_class_dir(root):
    for current_root, dirs, _ in os.walk(root):
        if EXPECTED_CLASSES.issubset(set(dirs)):
            return current_root
    raise RuntimeError(f"Could not find EuroSAT class folders under {root}")

CLASS_DIR = find_class_dir(EXTRACT_DIR)
label_names = sorted(EXPECTED_CLASSES)
num_classes = len(label_names)

records = []

for class_id, class_name in enumerate(label_names):
    class_path = os.path.join(CLASS_DIR, class_name)

    for filename in sorted(os.listdir(class_path)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff")):
            records.append(
                (os.path.join(class_path, filename), class_id)
            )

records = sorted(records, key=lambda x: x[0])

paths = np.array([r[0] for r in records], dtype=object)
labels = np.array([r[1] for r in records], dtype=np.int64)

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(
    len(paths),
    size=N_SAMPLES,
    replace=False,
)

sample_paths = paths[sample_idx]
y = labels[sample_idx]

idx_all = np.arange(N_SAMPLES)

idx_dev, idx_test = train_test_split(
    idx_all,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

idx_train, idx_val = train_test_split(
    idx_dev,
    test_size=0.125,
    stratify=y[idx_dev],
    random_state=SEED,
)

X_img = np.empty(
    (N_SAMPLES, 64, 64, 3),
    dtype=np.uint8,
)

for i, path in enumerate(sample_paths):
    with Image.open(path) as image:
        X_img[i] = np.asarray(
            image.convert("RGB"),
            dtype=np.uint8,
        )

X_train = X_img[idx_train]
X_val = X_img[idx_val]
X_test = X_img[idx_test]

y_train = y[idx_train]
y_val = y[idx_val]
y_test = y[idx_test]

print(
    f"train={len(y_train):,}  "
    f"validation={len(y_val):,}  "
    f"test={len(y_test):,}"
)
print("Classes:", label_names)

## 2) System A — human-designed features

This is the same representation logic from `06_01`.

Each image becomes 34 named features:

- 6 color statistics;
- 24 histogram values;
- 4 Sobel edge summaries.

The random forest never sees the original image.

It sees only those 34 measurements.

In [ ]:
SOBEL_X = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

SOBEL_Y = SOBEL_X.T


def color_stats(img_batch):
    x = img_batch.astype(np.float32) / 255.0

    means = x.mean(axis=(1, 2))
    stds = x.std(axis=(1, 2))

    return np.concatenate(
        [means, stds],
        axis=1,
    )


def color_histograms(img_batch, bins=8):
    n = img_batch.shape[0]

    output = np.zeros(
        (n, 3 * bins),
        dtype=np.float32,
    )

    edges = np.linspace(
        0,
        256,
        bins + 1,
    )

    for i in range(n):
        for channel in range(3):
            hist, _ = np.histogram(
                img_batch[i, ..., channel],
                bins=edges,
            )

            output[
                i,
                channel * bins : (channel + 1) * bins,
            ] = hist / hist.sum()

    return output


def edge_summaries(img_batch):
    gray = (
        img_batch.astype(np.float32).mean(axis=3)
        / 255.0
    )

    output = np.zeros(
        (gray.shape[0], 4),
        dtype=np.float32,
    )

    half = gray.shape[1] // 2

    for i in range(gray.shape[0]):
        gx = np.abs(
            convolve(
                gray[i],
                SOBEL_X,
                mode="nearest",
            )
        )

        gy = np.abs(
            convolve(
                gray[i],
                SOBEL_Y,
                mode="nearest",
            )
        )

        output[i] = [
            gx[:half].mean(),
            gy[:half].mean(),
            gx[half:].mean(),
            gy[half:].mean(),
        ]

    return output


def classical_features(img_batch):
    return np.concatenate(
        [
            color_stats(img_batch),
            color_histograms(img_batch),
            edge_summaries(img_batch),
        ],
        axis=1,
    )


X_train_classical = classical_features(X_train)
X_test_classical = classical_features(X_test)

system_a = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
)

system_a.fit(
    X_train_classical,
    y_train,
)

probs_a = system_a.predict_proba(
    X_test_classical
)

pred_a = probs_a.argmax(axis=1)

acc_a = accuracy_score(
    y_test,
    pred_a,
)

print(
    f"System A clean-test accuracy: "
    f"{acc_a:.3f}"
)

## 3) System B — pretrained visual representation

System B uses **MobileNetV2 pretrained on ImageNet**.

We remove the original ImageNet classifier and use the frozen network as a feature extractor.

Each image becomes a learned 1,280-dimensional vector.

Then we train ordinary logistic regression on those vectors.

Notice the contrast:

### System A

The dimensions were chosen by people:

```text
mean red
green histogram bin 3
edge strength
...
```

### System B

The dimensions were learned by a neural network from a much larger visual task.

We cannot attach a simple human label to each dimension.

This is transfer learning as **feature extraction**.

In [ ]:
BACKBONE_SIZE = (96, 96)

backbone = tf.keras.applications.MobileNetV2(
    input_shape=BACKBONE_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    pooling="avg",
)

backbone.trainable = False

preprocess = tf.keras.applications.mobilenet_v2.preprocess_input


def pretrained_features(img_batch):
    ds = tf.data.Dataset.from_tensor_slices(
        img_batch
    )

    ds = ds.map(
        lambda x: preprocess(
            tf.image.resize(
                tf.cast(x, tf.float32),
                BACKBONE_SIZE,
            )
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    ds = (
        ds
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return backbone.predict(
        ds,
        verbose=0,
    )


print("Extracting pretrained features...")

X_train_pretrained = pretrained_features(
    X_train
)

X_test_pretrained = pretrained_features(
    X_test
)

print(
    "Pretrained feature matrix:",
    X_train_pretrained.shape,
)

In [ ]:
system_b = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1500,
        C=1.0,
    ),
)

system_b.fit(
    X_train_pretrained,
    y_train,
)

probs_b = system_b.predict_proba(
    X_test_pretrained
)

pred_b = probs_b.argmax(axis=1)

acc_b = accuracy_score(
    y_test,
    pred_b,
)

benchmark = pd.DataFrame(
    {
        "system": [
            "System A: hand features + RF",
            "System B: pretrained features + logistic",
        ],
        "clean_test_accuracy": [
            acc_a,
            acc_b,
        ],
    }
)

benchmark

## Your turn: interpret the first result

### Question 2

Which system performs better?

Explain the result using the idea of **representation** rather than simply saying that one algorithm is more advanced.

**Your answer:**

> Replace this text with 2–4 sentences.

## 4) Where does each system fail?

Overall accuracy is only one number.

We want to know whether the systems fail in the same places.

First, inspect per-class performance.

In [ ]:
report_a = classification_report(
    y_test,
    pred_a,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)

report_b = classification_report(
    y_test,
    pred_b,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)

class_results = pd.DataFrame(
    {
        "class": label_names,
        "A_recall": [
            report_a[name]["recall"]
            for name in label_names
        ],
        "B_recall": [
            report_b[name]["recall"]
            for name in label_names
        ],
    }
)

class_results["B_minus_A"] = (
    class_results["B_recall"]
    - class_results["A_recall"]
)

class_results.sort_values(
    "B_minus_A"
)

### Question 3

Identify:

1. one class where the pretrained representation helps substantially;
2. one class that remains difficult even for System B.

What visual distinction do you think is causing the difficulty?

**Your answer:**

> Replace this text with your interpretation.

### Confusion matrices

The off-diagonal cells tell us *which wrong class* receives the image.

Look for a recurring pair rather than a single isolated error.

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(
        y_true,
        y_pred,
    )

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    im = ax.imshow(
        cm,
        cmap="Blues",
    )

    ax.set_xticks(range(num_classes))
    ax.set_xticklabels(
        label_names,
        rotation=90,
        fontsize=9,
    )

    ax.set_yticks(range(num_classes))
    ax.set_yticklabels(
        label_names,
        fontsize=9,
    )

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)

    plt.colorbar(
        im,
        ax=ax,
        fraction=0.046,
    )

    plt.tight_layout()
    plt.show()


plot_confusion(
    y_test,
    pred_a,
    "System A confusion matrix",
)

plot_confusion(
    y_test,
    pred_b,
    "System B confusion matrix",
)

### Question 4

Name one meaningful confusion pair for each system.

Do the two systems make the same kind of mistake?

**Your answer:**

> Replace this text with your answer.

## 5) Disagreement analysis

Two models can have similar accuracy and still make different decisions.

That matters operationally.

We will divide the test images into four groups:

- both correct;
- only A correct;
- only B correct;
- both wrong.

In [ ]:
a_correct = pred_a == y_test
b_correct = pred_b == y_test

comparison = pd.Series(
    {
        "Both correct": np.sum(
            a_correct & b_correct
        ),
        "Only A correct": np.sum(
            a_correct & ~b_correct
        ),
        "Only B correct": np.sum(
            ~a_correct & b_correct
        ),
        "Both wrong": np.sum(
            ~a_correct & ~b_correct
        ),
    }
)

comparison

### Inspect cases where the systems disagree

These are useful because the same image produced different conclusions from different representations.

In [ ]:
disagree = np.where(
    pred_a != pred_b
)[0]

# Put the highest combined-confidence disagreements first.
score = (
    probs_a.max(axis=1)
    + probs_b.max(axis=1)
)

disagree = disagree[
    np.argsort(
        -score[disagree]
    )
]

selected = disagree[:8]

fig, axs = plt.subplots(
    2,
    4,
    figsize=(12, 6),
)

for ax in axs.flat:
    ax.axis("off")

for i, ax in zip(
    selected,
    axs.flat,
):
    ax.imshow(
        X_test[i]
    )

    ax.set_title(
        f"true: {label_names[y_test[i]]}\n"
        f"A: {label_names[pred_a[i]]}  "
        f"B: {label_names[pred_b[i]]}",
        fontsize=9,
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

### Question 5

Look at the disagreement examples.

Do you see evidence that one representation is using information the other is missing?

Give one example.

**Your answer:**

> Replace this text with your answer.

# Part II — Stress test the systems

Clean test accuracy assumes future images resemble the test set.

Real systems encounter variation.

For satellite images, the underlying land-cover category should not change just because an image is:

- darker;
- slightly blurred;
- rotated 90 degrees.

Those transformations give us a simple robustness test.

> A robust classifier should ideally preserve its decision when the transformation preserves the underlying class.

## 6) Create three altered test sets

We will test:

### Darker

Pixel intensities are reduced to 55% of their original value.

### Blur

A modest Gaussian blur removes some fine detail.

### Rotate 90°

For overhead satellite imagery, a 90-degree rotation should not change the land-cover label.

In [ ]:
def darken(images, factor=0.55):
    x = (
        images.astype(np.float32)
        * factor
    )

    return np.clip(
        x,
        0,
        255,
    ).astype(np.uint8)


def blur(images, sigma=1.2):
    x = gaussian_filter(
        images.astype(np.float32),
        sigma=(0, sigma, sigma, 0),
    )

    return np.clip(
        x,
        0,
        255,
    ).astype(np.uint8)


def rotate_90(images):
    return np.rot90(
        images,
        k=1,
        axes=(1, 2),
    ).copy()


stress_sets = {
    "clean": X_test,
    "dark": darken(X_test),
    "blur": blur(X_test),
    "rotate90": rotate_90(X_test),
}

### See what the transformations actually do

In [ ]:
example = 0

fig, axs = plt.subplots(
    1,
    4,
    figsize=(12, 3),
)

for ax, (name, images) in zip(
    axs,
    stress_sets.items(),
):
    ax.imshow(
        images[example]
    )
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7) Re-evaluate both systems under stress

This cell recomputes each representation for the altered images.

That matters.

System A gets new hand-engineered measurements.

System B gets new pretrained feature vectors.

Then both classifiers make predictions using the models trained on the original clean training set.

In [ ]:
stress_results = []

stress_cache = {}

for condition, images in stress_sets.items():
    # System A: recompute human-designed features.
    a_features = classical_features(
        images
    )

    a_probs = system_a.predict_proba(
        a_features
    )

    a_pred = a_probs.argmax(axis=1)

    # System B: recompute pretrained visual features.
    b_features = pretrained_features(
        images
    )

    b_probs = system_b.predict_proba(
        b_features
    )

    b_pred = b_probs.argmax(axis=1)

    stress_cache[condition] = {
        "a_probs": a_probs,
        "a_pred": a_pred,
        "b_probs": b_probs,
        "b_pred": b_pred,
    }

    stress_results.append(
        {
            "condition": condition,
            "A_accuracy": accuracy_score(
                y_test,
                a_pred,
            ),
            "B_accuracy": accuracy_score(
                y_test,
                b_pred,
            ),
        }
    )

stress_df = pd.DataFrame(
    stress_results
)

clean_a = stress_df.loc[
    stress_df["condition"] == "clean",
    "A_accuracy",
].iloc[0]

clean_b = stress_df.loc[
    stress_df["condition"] == "clean",
    "B_accuracy",
].iloc[0]

stress_df["A_drop"] = (
    clean_a
    - stress_df["A_accuracy"]
)

stress_df["B_drop"] = (
    clean_b
    - stress_df["B_accuracy"]
)

stress_df

### Question 6

Which transformation hurts each system the most?

Explain why the result might follow from the way each system represents an image.

**Your answer:**

> Replace this text with 3–5 sentences.

## 8) Stability: does the prediction itself change?

Accuracy tells us whether the transformed image is correct.

A second question is whether the model changes its decision at all.

For each stress condition, calculate the percentage of predictions that remain the same as the model's clean-image prediction.

In [ ]:
clean_pred_a = stress_cache[
    "clean"
]["a_pred"]

clean_pred_b = stress_cache[
    "clean"
]["b_pred"]

stability_rows = []

for condition in stress_sets:
    a_pred = stress_cache[
        condition
    ]["a_pred"]

    b_pred = stress_cache[
        condition
    ]["b_pred"]

    stability_rows.append(
        {
            "condition": condition,
            "A_same_prediction": np.mean(
                a_pred == clean_pred_a
            ),
            "B_same_prediction": np.mean(
                b_pred == clean_pred_b
            ),
        }
    )

pd.DataFrame(
    stability_rows
)

### Question 7

Does the system with the highest clean accuracy also have the highest prediction stability?

Why is that distinction operationally important?

**Your answer:**

> Replace this text with your answer.

# Part III — When should a human review the prediction?

Suppose the company will automatically accept high-confidence classifications.

Lower-confidence cases go to a human analyst.

This creates a tradeoff:

- **higher threshold** → fewer automatic decisions, usually greater accuracy among them;
- **lower threshold** → more automation, but more risk.

This is a human–AI system design decision, not just a model metric.

In [ ]:
def threshold_table(
    probabilities,
    y_true,
    thresholds=(0.50, 0.60, 0.70, 0.80, 0.90),
):
    predictions = probabilities.argmax(
        axis=1
    )

    confidence = probabilities.max(
        axis=1
    )

    rows = []

    for threshold in thresholds:
        automated = (
            confidence >= threshold
        )

        coverage = automated.mean()

        if automated.any():
            accepted_accuracy = np.mean(
                predictions[automated]
                == y_true[automated]
            )
        else:
            accepted_accuracy = np.nan

        rows.append(
            {
                "threshold": threshold,
                "automation_rate": coverage,
                "human_review_rate": 1 - coverage,
                "accuracy_when_automated": accepted_accuracy,
            }
        )

    return pd.DataFrame(rows)


print("SYSTEM A")
display(
    threshold_table(
        probs_a,
        y_test,
    ).round(3)
)

print("\nSYSTEM B")
display(
    threshold_table(
        probs_b,
        y_test,
    ).round(3)
)

## The business requirement

The operations team gives you the following requirement:

> **Among classifications that are automated, we want at least 95% accuracy.**

Human review is expensive, so among thresholds that satisfy that requirement, prefer the one that automates the largest share of images.

### Question 8

For each system:

1. Is there a threshold that satisfies the 95% requirement?
2. If so, which threshold would you choose?
3. What percentage of images would still require human review?

**Your answer:**

> Replace this text with your answer.

## 9) One final complication: confidence is not certainty

A probability such as `0.92` is a model output.

It is **not** a guarantee that the prediction is correct.

Confidence can become unreliable when the input distribution changes—the exact issue we just explored with darkened, blurred, and rotated images.

### Question 9

Why might a confidence threshold chosen on clean test images fail when the production images change?

What would you monitor after deployment?

**Your answer:**

> Replace this text with your answer.

# Final recommendation

You now have more evidence than a simple leaderboard.

Your recommendation should consider:

- clean accuracy;
- which classes are difficult;
- disagreements between systems;
- robustness;
- prediction stability;
- confidence threshold;
- human-review workload.

## Question 10 — Decision memo

Write **150–250 words** to the operations lead.

Recommend:

1. **System A or System B** as the primary classifier;
2. a confidence threshold or human-review policy;
3. at least one class or condition that deserves special monitoring;
4. one limitation that prevents you from treating the system as fully autonomous.

**Your recommendation:**

> Replace this text with your memo.

# Optional extension — What changes with a vision-language foundation model?

The systems in this exercise still require a fixed supervised classification task.

Even System B needs labeled EuroSAT examples so logistic regression can learn the 10 categories.

A vision-language foundation model changes the interface.

Instead of training a new classifier, we might compare an image directly with text prompts such as:

```text
"a satellite image of a forest"
"a satellite image of a highway"
"a satellite image of an industrial area"
"a satellite image of residential buildings"
```

That could make **zero-shot classification** possible.

But flexibility introduces new questions:

- How sensitive is the result to prompt wording?
- Are the categories represented well in the model's pretraining data?
- Is zero-shot performance good enough for this domain?
- How should confidence be interpreted?
- Does the model behave differently across geographic regions or sensor conditions?

### Optional question

Would you replace System B immediately with a zero-shot vision-language model?

Or would you treat it as another candidate system to validate?

Explain why.

**Your answer:**

> Replace this text with your answer.

## What this exercise was really about

The module began with a technical question:

> **How do computers represent images?**

The exercise ends with a systems question:

> **When is a vision model good enough to act, and when should a human remain in the loop?**

The progression matters:

\[
\text{pixels}
\rightarrow
\text{human features}
\rightarrow
\text{learned features}
\rightarrow
\text{pretrained features}
\rightarrow
\text{foundation-model representations}
\]

But better representation does not eliminate the need to evaluate:

- errors;
- robustness;
- confidence;
- deployment context.

That is the difference between **building a model** and **designing an AI system**.